# Risk-Controlled Momentum: Day 1 Prototype

This notebook develops the pre-registered model sequentially. Each section should be understood and checked before proceeding.

## 1. Imports and configuration

### Concept before code

An **import** makes a library's tools available in the notebook. The short names are conventional aliases: `np` for NumPy, `pd` for pandas, `plt` for Matplotlib plotting, `sns` for seaborn and `yf` for yfinance. An alias changes only how we refer to a library; it does not change the library.

A Python **dictionary** stores labelled key-value pairs. Keeping every pre-registered choice in `LOCKED_CONFIG` separates research decisions from later calculations and makes accidental parameter changes easier to notice. Percentages are decimals in calculations: 10% is `0.10`. One basis point is one ten-thousandth, so 10 basis points is `10 / 10_000 = 0.001`.

The one-day `timing_lag_days` value records the rule that a return on day `t` can use information available no later than day `t-1`. It does not perform the lag yet; the lag will be implemented and checked in the signal and volatility sections.

The final two lines turn the dictionary into a one-column table for visual inspection. Nothing in this section downloads data or calculates a strategy result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf

LOCKED_CONFIG = {
    "ticker": "SPY",
    "sample_start": "2007-01-01",
    "sample_end": "2025-12-31",
    "frequency": "daily",
    "price_field": "adjusted",
    "return_type": "simple",
    "momentum_lookback_days": 252,
    "volatility_window_days": 21,
    "annualisation_days": 252,
    "target_volatility": 0.10,
    "maximum_exposure": 1.5,
    "rebalancing": "daily",
    "transaction_cost_bps": 10,
    "transaction_cost_rate": 10 / 10_000,
    "cash_return": 0.0,
    "timing_lag_days": 1,
}

config_table = pd.Series(LOCKED_CONFIG, name="Locked Day 1 value").to_frame()
config_table

## 2. Download and validate adjusted price data

### Concept before code

A **DataFrame** is a labelled table with rows and columns. `yf.download(...)` returns a DataFrame containing daily market fields. A **Series** is one labelled column; after inspecting the download, we select only the adjusted SPY closing-price series needed by this model.

SPY pays distributions. An unadjusted close can fall when cash leaves the fund even though an investor received that cash. Because the locked specification requires an adjusted price, we explicitly set `auto_adjust=True`. In installed `yfinance 1.6.0`, this adjusts the OHLC fields and places the adjusted closing price in `Close`; a separate `Adj Close` column is therefore absent.

The provider treats `start` as inclusive and `end` as exclusive. We convert the locked end-date string to a pandas `Timestamp`, add a one-day `Timedelta`, and convert it back to a date string. Requesting an exclusive boundary of `2026-01-01` is how we include the locked final date `2025-12-31`; it does not extend the research sample.

The validation dictionary stores named True/False checks. We require a datetime index, increasing and unique dates, no missing selected prices, positive prices, and coverage contained within both ends of the locked sample. The seven-calendar-day tolerance recognises that 1 January or 31 December can be a weekend or market holiday; it is a data-coverage check, not a model parameter. If any check is False, `raise ValueError(...)` stops the notebook instead of allowing bad data to flow into the strategy.

In [ ]:
sample_start = pd.Timestamp(LOCKED_CONFIG["sample_start"])
sample_end = pd.Timestamp(LOCKED_CONFIG["sample_end"])
download_end_exclusive = (sample_end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

raw_spy = yf.download(
    tickers=LOCKED_CONFIG["ticker"],
    start=LOCKED_CONFIG["sample_start"],
    end=download_end_exclusive,
    interval="1d",
    auto_adjust=True,
    actions=False,
    keepna=False,
    progress=False,
    multi_level_index=False,
)

if raw_spy.empty:
    raise RuntimeError("SPY download returned no observations.")

if "Close" not in raw_spy.columns:
    raise ValueError(f"Adjusted Close field not found. Returned columns: {raw_spy.columns.tolist()}")

spy_price = raw_spy["Close"].rename("adjusted_price")

validation_checks = {
    "datetime_index": isinstance(spy_price.index, pd.DatetimeIndex),
    "dates_in_increasing_order": spy_price.index.is_monotonic_increasing,
    "dates_are_unique": not spy_price.index.has_duplicates,
    "no_missing_prices": not spy_price.isna().any(),
    "all_prices_are_positive": spy_price.gt(0).all(),
    "starts_inside_locked_sample": spy_price.index.min() >= sample_start,
    "ends_inside_locked_sample": spy_price.index.max() <= sample_end,
    "covers_locked_sample_start": spy_price.index.min() <= sample_start + pd.Timedelta(days=7),
    "covers_locked_sample_end": spy_price.index.max() >= sample_end - pd.Timedelta(days=7),
}

failed_checks = [name for name, passed in validation_checks.items() if not passed]
if failed_checks:
    raise ValueError(f"SPY data validation failed: {failed_checks}")

data_summary = pd.Series(
    {
        "ticker": LOCKED_CONFIG["ticker"],
        "selected_field": spy_price.name,
        "first_observation": spy_price.index.min().date(),
        "last_observation": spy_price.index.max().date(),
        "observations": spy_price.size,
        "missing_prices": int(spy_price.isna().sum()),
        "download_end_exclusive": download_end_exclusive,
    },
    name="Validated value",
)

display(data_summary.to_frame())
display(pd.Series(validation_checks, name="Passed").to_frame())
display(spy_price.head(3).to_frame())
display(spy_price.tail(3).to_frame())

## 3. Calculate simple daily returns

## 4. Build the buy-and-hold benchmark

## 5. Build and lag the momentum signal

## 6. Estimate and lag rolling volatility

## 7. Apply volatility targeting and the leverage cap

## 8. Calculate turnover and transaction costs

## 9. Calculate performance metrics

## 10. Visualise preliminary results

## 11. Run sanity checks and record observations